In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
ls = pd.read_csv(Path("../data/raw/ls_ghep.csv"))

In [ ]:
KEEP_COLS = {
    # Định danh, thời gian, không gian & kiểm tra tự động
    "id", "last_scraped", "room_type", "neighbourhood_cleansed",
    "latitude", "longitude",
    # Giá & sức chứa
    "price", "accommodates", "minimum_nights", "maximum_nights",
    # Tiện ích & phòng ốc
    "amenities", "bathrooms", "bathrooms_text", "bedrooms", "beds",
    # Đánh giá
    "number_of_reviews", "review_scores_rating", "review_scores_value",
    # Chủ nhà & độ sẵn sàng
    "host_id", "calculated_host_listings_count",
    "has_availability", "availability_365",
}

In [ ]:
#Kiểm tra cấu trúc dữ liệu đầu vào
def kiem_cau_truc(d: pd.DataFrame, moc: str = "") -> None:
    #Kiểm tra đủ cột KEEP_COLS và các cột số đúng kiểu số.
    ten_moc = f" [{moc}]" if moc else ""
    thieu = KEEP_COLS - set(d.columns)
    if thieu:
        raise ValueError(f"Thiếu cột ở mốc{ten_moc}: {sorted(thieu)}")
    for cot in ["minimum_nights", "maximum_nights", "review_scores_rating",
                "accommodates", "latitude", "longitude"]:
        if not pd.api.types.is_numeric_dtype(d[cot]):
            raise TypeError(f"Cột {cot} (mốc {ten_moc}) phải có kiểu số, kiểu hiện tại là {d[cot].dtype}")

In [ ]:
#gán cờ giá ngoại lai ở mốc cuối
def gia_ngoai_lai(df, last_snap):
    mask_last_price = (df["snapshot"] == last_snap) & df["price_clean"].notna() & (df["price_clean"] > 0)
    q1, q3 = df.loc[mask_last_price, "price_clean"].quantile([0.25, 0.75])
    iqr = q3 - q1
    price_iqr_low, price_iqr_high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    df["flag_outlier_price_iqr"] = (
        mask_last_price
        & ((df["price_clean"] < price_iqr_low) | (df["price_clean"] > price_iqr_high))
    )


In [ ]:
def dien_khuyet_gia(df, last_snap):
    mask_last = df["snapshot"] == last_snap
    df["price_clean_fillna"] = np.nan
    df.loc[mask_last, "price_clean_fillna"] = (
        df[mask_last].groupby(["room_type", "accommodates"])["price_clean"]
        .transform(lambda s: s.fillna(s.median()))
    )
    # Nhóm không có giá nào -> dùng trung vị toàn mốc cuối
    df.loc[mask_last, "price_clean_fillna"] = df.loc[mask_last, "price_clean_fillna"].fillna(
        df.loc[mask_last, "price_clean"].median()
    )
